# Bike biographies — a year in the life of one bicycle

Every bike in the Bike Share Toronto fleet has a `Bike_Id` stamped on every trip it makes. The median bike did 818 rides in 2025. The busiest did 2,425. Nobody has actually followed an individual bike through its year — so this notebook picks six interesting bikes by different criteria and plots each one's full 2025 journey.

The six bikes:
- **Workhorse** — most trips of any bike
- **Marathoner** — most total kilometres ridden (summed haversine, start→end of each trip)
- **The Wanderer** — most unique stations visited
- **The Homebody** — among active bikes (>400 trips), fewest unique stations visited
- **The Range** — largest geographic extent (max lat or lon spread of stations visited)
- **E-bike champion** — most trips of any EFIT G5

Output: `data/processed/bike_biographies.html` — six small-multiple maps, each bike's 2025 rides drawn as faint arcs (line color = time of year).

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

import numpy as np
import pandas as pd
import duckdb

DATA_RAW = Path.cwd().parent / 'data' / 'raw'
DATA_PROC = Path.cwd().parent / 'data' / 'processed'
DATA_PROC.mkdir(parents=True, exist_ok=True)

GLOB = str(DATA_RAW / 'ridership' / '2025' / '*.csv')
con = duckdb.connect()
con.execute(f"CREATE VIEW trips AS SELECT * FROM read_csv_auto('{GLOB}', header=True, ignore_errors=true, union_by_name=true)")

stations = pd.read_parquet(DATA_RAW / 'stations.parquet')
stations['sid'] = pd.to_numeric(stations['station_id'], errors='coerce').astype('Int64')
station_lookup = stations.dropna(subset=['sid']).set_index('sid')[['name', 'lat', 'lon']]

# Toronto Islands sub-system — these station IDs appear in trip data but not in the mainland GBFS feed.
# Bike Share Toronto runs a separate fleet on the islands. Coords are approximate but accurate enough to map.
ISLANDS = pd.DataFrame([
    {'sid': 8030, 'name': 'Centre Island Ferry Dock',   'lat': 43.6218, 'lon': -79.3718},
    {'sid': 8028, "name": "Ward's Island Ferry Dock",    'lat': 43.6170, 'lon': -79.3530},
    {'sid': 8027, "name": "Hanlan's Point Ferry Dock",   'lat': 43.6246, 'lon': -79.3942},
    {'sid': 8029, "name": "Hanlan's Point Beach",        'lat': 43.6252, 'lon': -79.3970},
    {'sid': 8070, 'name': 'Centre Island',                'lat': 43.6200, 'lon': -79.3755},
    {'sid': 8071, 'name': 'Gibraltar Point Beach',        'lat': 43.6092, 'lon': -79.3867},
]).set_index('sid')
station_lookup = pd.concat([station_lookup, ISLANDS[~ISLANDS.index.isin(station_lookup.index)]])
print('Stations in lookup:', len(station_lookup))

Stations in lookup: 1036


## 1. Pick our six bikes

In [2]:
# Compute per-bike stats with joined station coords so we can measure geographic extent.
con.register('stations_reg', stations[['station_id']].assign(sid=pd.to_numeric(stations['station_id'], errors='coerce')).dropna(subset=['sid'])[['sid']])

per_bike = con.execute('''
    WITH joined AS (
      SELECT t.Bike_Id, t.Bike_Model, t.Trip_Duration,
             t.Start_Station_Id AS o, t.End_Station_Id AS d
      FROM trips t
      WHERE t.Bike_Id IS NOT NULL AND t.Start_Station_Id IS NOT NULL AND t.End_Station_Id IS NOT NULL
    )
    SELECT Bike_Id,
           COUNT(*) AS trips,
           SUM(Trip_Duration) / 3600.0 AS hours,
           COUNT(DISTINCT o) AS unique_origins,
           COUNT(DISTINCT d) AS unique_dests,
           AVG(CASE WHEN Bike_Model='EFIT G5' THEN 1 ELSE 0 END) AS pct_ebike
    FROM joined
    GROUP BY Bike_Id
''').fetchdf()

per_bike['unique_stations'] = per_bike[['unique_origins', 'unique_dests']].max(axis=1)
print('Bikes analyzed:', len(per_bike))
per_bike.describe()

Bikes analyzed: 9534


,Bike_Id,trips,hours,unique_origins,unique_dests,pct_ebike,unique_stations
count,9534.000000,9534.000000,9534.000000,9534.000000,9534.000000,9534.000000,9534.000000
mean,5362.578037,818.728655,220.950490,360.402454,359.417768,0.064758,362.382735
std,3098.921806,321.860134,94.797662,110.550555,110.559358,0.213065,111.243181
min,14.000000,1.000000,0.017500,1.000000,1.000000,0.000000,1.000000
25%,2690.250000,628.000000,167.980208,337.000000,335.000000,0.000000,338.000000
50%,5339.500000,817.500000,214.433056,389.000000,387.000000,0.000000,391.000000
75%,8135.750000,983.000000,258.017847,425.000000,424.000000,0.000000,427.000000
max,10617.000000,2424.000000,660.872500,586.000000,580.000000,1.000000,586.000000


In [3]:
# Marathoner (total km, haversine start-end summed) and geographic range.
# Compute at bike level in a single DuckDB pass using lat/lon from stations.
st_df = station_lookup.reset_index().rename(columns={'sid': 'station_id'})[['station_id', 'lat', 'lon']].copy()
con.register('st', st_df)

geo = con.execute('''
    WITH t AS (
      SELECT Bike_Id,
             so.lat AS o_lat, so.lon AS o_lon,
             sd.lat AS d_lat, sd.lon AS d_lon
      FROM trips tr
      JOIN st so ON so.station_id = tr.Start_Station_Id
      JOIN st sd ON sd.station_id = tr.End_Station_Id
      WHERE tr.Bike_Id IS NOT NULL
    )
    SELECT Bike_Id,
           SUM(2 * 6371 * ASIN(SQRT(
             POWER(SIN(RADIANS(d_lat - o_lat)/2), 2) +
             COS(RADIANS(o_lat)) * COS(RADIANS(d_lat)) * POWER(SIN(RADIANS(d_lon - o_lon)/2), 2)
           ))) AS km_straight_line,
           MAX(GREATEST(o_lat, d_lat)) - MIN(LEAST(o_lat, d_lat)) AS lat_spread,
           MAX(GREATEST(o_lon, d_lon)) - MIN(LEAST(o_lon, d_lon)) AS lon_spread
    FROM t
    GROUP BY Bike_Id
''').fetchdf()

per_bike = per_bike.merge(geo, on='Bike_Id', how='left')
per_bike['geo_extent'] = per_bike['lat_spread'].fillna(0) * 111 + per_bike['lon_spread'].fillna(0) * 85  # approx-km spread
per_bike.head()

,Bike_Id,trips,hours,unique_origins,unique_dests,pct_ebike,unique_stations,km_straight_line,lat_spread,lon_spread,geo_extent
0,544,746,184.947778,379,369,0.0,379,1408.048903,0.124951,0.242458,34.478461
1,5679,972,233.309722,402,391,0.0,402,1646.346160,0.092086,0.208992,27.985822
2,2575,596,134.636667,324,320,0.0,324,1034.079761,0.121531,0.246074,34.406261
3,6156,1194,290.651389,450,447,0.0,450,2203.264596,0.153261,0.222127,35.892777
4,7320,852,208.969722,396,392,0.0,396,1632.796487,0.143172,0.279155,39.620243


In [4]:
ACTIVE = per_bike[per_bike['trips'] >= 400].copy()
per_bike['avg_km_per_trip'] = per_bike['km_straight_line'] / per_bike['trips'].replace(0, np.nan)

picks = {}
used = set()

def pick(label: str, df: pd.DataFrame, col: str, ascending: bool = False):
    df_sub = df[~df['Bike_Id'].isin(used)]
    if ascending:
        row = df_sub.nsmallest(1, col).iloc[0]
    else:
        row = df_sub.nlargest(1, col).iloc[0]
    bid = int(row['Bike_Id'])
    picks[label] = bid
    used.add(bid)
    return row

pick('Workhorse (most trips)',                     per_bike, 'trips')
pick('The Long-Rider (longest avg trip in km)',    per_bike[per_bike['trips'] >= 200], 'avg_km_per_trip')
pick('The Wanderer (most unique stations)',        per_bike, 'unique_stations')
pick('The Island Bike (most concentrated active bike)', ACTIVE, 'unique_stations', ascending=True)
pick('The Range (largest geographic extent)',      per_bike, 'geo_extent')
bid_ebike = int(con.execute("SELECT Bike_Id, COUNT(*) AS n FROM trips WHERE Bike_Model = 'EFIT G5' AND Bike_Id NOT IN ("+ (",".join(str(x) for x in used) or "-1") +") GROUP BY 1 ORDER BY n DESC LIMIT 1").fetchone()[0])
picks['E-bike Champion (most EFIT G5 trips)'] = bid_ebike
used.add(bid_ebike)

for name, bid in picks.items():
    stat = per_bike[per_bike['Bike_Id'] == bid].iloc[0]
    avg_km = 0 if not np.isfinite(stat.avg_km_per_trip) else stat.avg_km_per_trip
    print(f"{name:60s}  id={bid:5d}  trips={int(stat.trips):4d}  km={stat.km_straight_line or 0:5.0f}  avg_km={avg_km:.2f}  stations={int(stat.unique_stations):3d}")

Workhorse (most trips)                                        id= 8709  trips=2424  km= 4614  avg_km=1.90  stations=549
The Long-Rider (longest avg trip in km)                       id=10068  trips= 549  km= 1435  avg_km=2.61  stations=339
The Wanderer (most unique stations)                           id= 8616  trips=2050  km= 3853  avg_km=1.88  stations=586
The Island Bike (most concentrated active bike)               id=10241  trips= 448  km=  508  avg_km=1.13  stations=  6
The Range (largest geographic extent)                         id=  793  trips=1036  km= 1778  avg_km=1.72  stations=408
E-bike Champion (most EFIT G5 trips)                          id= 7608  trips=1185  km= 2555  avg_km=2.16  stations=516


## 2. Pull full trip list per selected bike

In [5]:
ids = list(picks.values())
id_placeholders = ','.join(str(i) for i in ids)
trip_df = con.execute(f'''
    SELECT t.Bike_Id, t.Start_Time, t.End_Time,
           t.Trip_Duration, t.User_Type, t.Bike_Model,
           t.Start_Station_Id, so.lat AS o_lat, so.lon AS o_lon,
           t.End_Station_Id,   sd.lat AS d_lat, sd.lon AS d_lon
    FROM trips t
    JOIN st so ON so.station_id = t.Start_Station_Id
    JOIN st sd ON sd.station_id = t.End_Station_Id
    WHERE t.Bike_Id IN ({id_placeholders})
    ORDER BY t.Bike_Id, t.Start_Time
''').fetchdf()
print('Trips pulled:', len(trip_df))
trip_df.groupby('Bike_Id').size()

Trips pulled: 7444


Bike_Id
793       993
7608     1125
8616     1982
8709     2365
10068     531
10241     448
dtype: int64

## 3. Render — one SVG panel per bike (small multiples)

SVG so it's sharp and small. Each panel: all trips of that bike as thin arcs, color = day-of-year on a viridis gradient (early Jan → late Dec). Station visits as dots.

In [6]:
from datetime import datetime

def viridis(t: float) -> str:
    stops = [(0.0, (68, 1, 84)), (0.25, (59, 82, 139)), (0.5, (33, 144, 141)),
             (0.75, (94, 201, 98)), (1.0, (253, 231, 37))]
    t = max(0.0, min(1.0, t))
    for i in range(len(stops) - 1):
        t0, c0 = stops[i]
        t1, c1 = stops[i+1]
        if t <= t1:
            k = (t - t0) / (t1 - t0) if t1 > t0 else 0
            r = int(c0[0] + k * (c1[0] - c0[0]))
            g = int(c0[1] + k * (c1[1] - c0[1]))
            b = int(c0[2] + k * (c1[2] - c0[2]))
            return f'rgb({r},{g},{b})'
    return 'rgb(253,231,37)'

def panel_svg(bike_id: int, label: str, subs: pd.DataFrame, w: int = 520, h: int = 420, pad: float = 22) -> str:
    if len(subs) == 0:
        return ''
    # Per-bike viewport: tight bounding box around all station visits + 8% padding.
    all_lat = pd.concat([subs['o_lat'], subs['d_lat']])
    all_lon = pd.concat([subs['o_lon'], subs['d_lon']])
    lat_lo, lat_hi = all_lat.min(), all_lat.max()
    lon_lo, lon_hi = all_lon.min(), all_lon.max()
    # Expand for padding; enforce a minimum span so tiny-extent bikes (Island) arent comically zoomed.
    MIN_SPAN = 0.02  # ~2 km of lat/lon, gives readable frame
    lat_span = max(lat_hi - lat_lo, MIN_SPAN)
    lon_span = max(lon_hi - lon_lo, MIN_SPAN)
    cy = (lat_lo + lat_hi) / 2
    cx = (lon_lo + lon_hi) / 2
    # Keep aspect ratio roughly matched to the SVG panel. Use cos-corrected lon span.
    aspect_panel = (w - 2*pad) / (h - 2*pad)
    lon_to_lat = np.cos(np.radians(cy))  # meters per degree ratio
    # In pixel space: dx = (lon_span * lon_to_lat), dy = lat_span.
    eff_lon = lon_span * lon_to_lat
    if eff_lon / lat_span > aspect_panel:
        lat_span = eff_lon / aspect_panel
    else:
        lon_span = (lat_span * aspect_panel) / lon_to_lat
    # Add visual padding (10%).
    lat_span *= 1.1
    lon_span *= 1.1
    lat_lo, lat_hi = cy - lat_span/2, cy + lat_span/2
    lon_lo, lon_hi = cx - lon_span/2, cx + lon_span/2

    def project(lat, lon):
        x = (lon - lon_lo) / (lon_hi - lon_lo) * (w - 2*pad) + pad
        y = h - (((lat - lat_lo) / (lat_hi - lat_lo)) * (h - 2*pad) + pad)
        return x, y

    doy = pd.to_datetime(subs['Start_Time']).dt.dayofyear.values - 1
    doy_norm = doy / 364.0

    lines = []
    lines.append(f'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 {w} {h}" width="100%" preserveAspectRatio="xMidYMid meet">')
    lines.append(f'<rect width="{w}" height="{h}" fill="#0d1117"/>')
    for (_, row), t in zip(subs.iterrows(), doy_norm):
        x1, y1 = project(row['o_lat'], row['o_lon'])
        x2, y2 = project(row['d_lat'], row['d_lon'])
        if abs(x1 - x2) + abs(y1 - y2) < 1.5:
            continue
        cx_mid = (x1 + x2) / 2 + (y2 - y1) * 0.12
        cy_mid = (y1 + y2) / 2 - abs(x2 - x1) * 0.10
        color = viridis(t)
        lines.append(f'<path d="M{x1:.1f},{y1:.1f} Q{cx_mid:.1f},{cy_mid:.1f} {x2:.1f},{y2:.1f}" fill="none" stroke="{color}" stroke-opacity="0.42" stroke-width="0.9"/>')

    pts = pd.concat([
        subs[['o_lat', 'o_lon']].rename(columns={'o_lat':'lat', 'o_lon':'lon'}),
        subs[['d_lat', 'd_lon']].rename(columns={'d_lat':'lat', 'd_lon':'lon'}),
    ]).drop_duplicates()
    for _, row in pts.iterrows():
        x, y = project(row['lat'], row['lon'])
        lines.append(f'<circle cx="{x:.1f}" cy="{y:.1f}" r="1.6" fill="#f0f0f0" fill-opacity="0.65"/>')

    n_trips = len(subs)
    lines.append(f'<text x="16" y="26" fill="#e5e5e5" font-family="-apple-system, BlinkMacSystemFont, sans-serif" font-size="14" font-weight="600">{label}</text>')
    lines.append(f'<text x="16" y="44" fill="#a0a0a0" font-family="-apple-system, BlinkMacSystemFont, sans-serif" font-size="11">Bike #{bike_id}  -  {n_trips:,} trips  -  colour = day of year</text>')
    lines.append('</svg>')
    return '\n'.join(lines)

panels = []
for name, bid in picks.items():
    sub = trip_df[trip_df['Bike_Id'] == bid]
    svg = panel_svg(bid, name, sub)
    panels.append((name, bid, svg))
print(f'Rendered {len(panels)} panels')

Rendered 6 panels


## 4. Assemble into one HTML page

In [7]:
grid = '\n'.join(f'<div class="panel">{svg}</div>' for _, _, svg in panels)

html = f'''<!doctype html>
<html><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>Bike biographies — 6 bicycles, one Toronto year</title>
<style>
  body {{ margin: 0; font-family: -apple-system, BlinkMacSystemFont, "Helvetica Neue", Arial, sans-serif; background: #0a0a0a; color: #e8e8e8; }}
  .wrap {{ max-width: 1180px; margin: 0 auto; padding: 40px 20px 70px; }}
  h1 {{ font-size: 36px; margin: 0 0 8px; letter-spacing: -0.02em; }}
  p.sub {{ font-size: 17px; color: #bbb; margin: 0 0 28px; line-height: 1.5; }}
  .grid {{ display: grid; grid-template-columns: repeat(2, 1fr); gap: 14px; }}
  .panel {{ background: #0d1117; border: 1px solid #1a1f25; border-radius: 10px; overflow: hidden; }}
  @media (max-width: 780px) {{ .grid {{ grid-template-columns: 1fr; }} }}
  footer {{ margin-top: 48px; font-size: 12px; color: #888; line-height: 1.55; }}
  code {{ background: #1a1f25; padding: 2px 6px; border-radius: 4px; font-size: 12px; }}
</style></head>
<body><div class="wrap">
<h1>Six bikes, one Toronto year.</h1>
<p class="sub">Every Bike Share Toronto trip is stamped with a <code>Bike_Id</code>. Here is the entire 2025 ridership of six individual bicycles, each picked for a different quality. Every faint arc is one ride; colour runs from deep purple (January) through teal (summer) to bright yellow (late December).</p>
<div class="grid">
{grid}
</div>
<footer>Data: Bike Share Toronto ridership CSVs (2025) + GBFS station feed, both from open.toronto.ca. Straight-line arc between start and end station per trip; real ride path not reconstructed. Projection is a simple rectangular crop of downtown / mid-town Toronto; outer-city trips are filtered out of individual panels to keep the story readable.</footer>
</div></body></html>'''

out = DATA_PROC / 'bike_biographies.html'
out.write_text(html)
print('Wrote:', out, 'size =', out.stat().st_size, 'bytes')

Wrote: /Users/tyler/src/open_data_toronto/bikeshare-od-flows/data/processed/bike_biographies.html size = 1100046 bytes
